# **Dataset Understanding**
The dataset used in this project contains medical information of individuals and is used to predict whether a person has diabetes or not. Each row represents a different patient, and each column represents a specific health-related feature such as age, BMI, glucose level, and medical history.

The target variable in this dataset is “diabetes”, which has two values:

0 → Person does not have diabetes
1 → Person has diabetes

The dataset includes both numerical and categorical features, which are useful for building a machine learning classification model.

***Columns Explanation***

gender: Represents the gender of the person (Male/Female)

age: Age of the person

hypertension: Whether the person has high blood pressure (0 = No, 1 = Yes)

heart_disease: Whether the person has heart disease

smoking_history: Smoking behavior of the person

bmi: Body Mass Index (health indicator)

HbA1c_level: Average blood sugar level over time

blood_glucose_level: Current glucose level in blood


 **Target variable**

diabetes: Target variable (0 or 1)

# Create Spark Session

In [3]:
from pyspark.sql import SparkSession
session=SparkSession.builder.appName("Diabetes Prediction").getOrCreate()

# Load Data in PySpark

In [4]:
df=session.read.csv("diabetes_prediction_dataset.csv",header=True,inferSchema=True)
df.show(5)

+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
|gender| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|diabetes|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
|Female|80.0|           0|            1|          never|25.19|        6.6|                140|       0|
|Female|54.0|           0|            0|        No Info|27.32|        6.6|                 80|       0|
|  Male|28.0|           0|            0|          never|27.32|        5.7|                158|       0|
|Female|36.0|           0|            0|        current|23.45|        5.0|                155|       0|
|  Male|76.0|           1|            1|        current|20.14|        4.8|                155|       0|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
only showing top 5 rows


# EDA (Exploratory Data Analysis)

In [5]:
df.printSchema()

root
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- heart_disease: integer (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- HbA1c_level: double (nullable = true)
 |-- blood_glucose_level: integer (nullable = true)
 |-- diabetes: integer (nullable = true)



In [6]:
df.dtypes

[('gender', 'string'),
 ('age', 'double'),
 ('hypertension', 'int'),
 ('heart_disease', 'int'),
 ('smoking_history', 'string'),
 ('bmi', 'double'),
 ('HbA1c_level', 'double'),
 ('blood_glucose_level', 'int'),
 ('diabetes', 'int')]

In [7]:
df.describe().show()

+-------+------+-----------------+------------------+------------------+---------------+-----------------+------------------+-------------------+-------------------+
|summary|gender|              age|      hypertension|     heart_disease|smoking_history|              bmi|       HbA1c_level|blood_glucose_level|           diabetes|
+-------+------+-----------------+------------------+------------------+---------------+-----------------+------------------+-------------------+-------------------+
|  count|100000|           100000|            100000|            100000|         100000|           100000|            100000|             100000|             100000|
|   mean|  NULL|41.88585600000013|           0.07485|           0.03942|           NULL|27.32076709999422|5.5275069999983275|          138.05806|              0.085|
| stddev|  NULL|22.51683987161704|0.2631504702289171|0.1945930169980986|           NULL|6.636783416648357|1.0706720918835468|  40.70813604870383|0.27888308976661896|
|   

In [8]:
df.count()

100000

In [9]:
df.columns

['gender',
 'age',
 'hypertension',
 'heart_disease',
 'smoking_history',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'diabetes']

In [10]:
df.groupBy("diabetes").count().show()

+--------+-----+
|diabetes|count|
+--------+-----+
|       1| 8500|
|       0|91500|
+--------+-----+



In [11]:
from pyspark.sql.functions import col,sum
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
 for c in df.columns]).show()

+------+---+------------+-------------+---------------+---+-----------+-------------------+--------+
|gender|age|hypertension|heart_disease|smoking_history|bmi|HbA1c_level|blood_glucose_level|diabetes|
+------+---+------------+-------------+---------------+---+-----------+-------------------+--------+
|     0|  0|           0|            0|              0|  0|          0|                  0|       0|
+------+---+------------+-------------+---------------+---+-----------+-------------------+--------+



# Preprocessing

In [12]:
df=df.dropna()

In [13]:
df=df.fillna({
    "bmi":0,
    "age":0,
    "gender":"unknown"
})
df.show()

+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
|gender| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|diabetes|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
|Female|80.0|           0|            1|          never|25.19|        6.6|                140|       0|
|Female|54.0|           0|            0|        No Info|27.32|        6.6|                 80|       0|
|  Male|28.0|           0|            0|          never|27.32|        5.7|                158|       0|
|Female|36.0|           0|            0|        current|23.45|        5.0|                155|       0|
|  Male|76.0|           1|            1|        current|20.14|        4.8|                155|       0|
|Female|20.0|           0|            0|          never|27.32|        6.6|                 85|       0|
|Female|44.0|           0|            0|          never|19.31|  

In [14]:
df.dropDuplicates()

DataFrame[gender: string, age: double, hypertension: int, heart_disease: int, smoking_history: string, bmi: double, HbA1c_level: double, blood_glucose_level: int, diabetes: int]

In [15]:
from pyspark.sql.functions import mean
mean_bmi=df.select(mean("bmi")).collect()[0][0]
df=df.fillna({
     "bmi":mean_bmi
})

In [16]:
from pyspark.ml.feature import StringIndexer
indexer=StringIndexer(
    inputCol="gender",
    outputCol="gender_index"
)
df=indexer.fit(df).transform(df)
df.show()

+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+------------+
|gender| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|diabetes|gender_index|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+------------+
|Female|80.0|           0|            1|          never|25.19|        6.6|                140|       0|         0.0|
|Female|54.0|           0|            0|        No Info|27.32|        6.6|                 80|       0|         0.0|
|  Male|28.0|           0|            0|          never|27.32|        5.7|                158|       0|         1.0|
|Female|36.0|           0|            0|        current|23.45|        5.0|                155|       0|         0.0|
|  Male|76.0|           1|            1|        current|20.14|        4.8|                155|       0|         1.0|
|Female|20.0|           0|            0|          never|27.32|  

In [17]:
df=df.drop("gender")
df.show()

+----+------------+-------------+---------------+-----+-----------+-------------------+--------+------------+
| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|diabetes|gender_index|
+----+------------+-------------+---------------+-----+-----------+-------------------+--------+------------+
|80.0|           0|            1|          never|25.19|        6.6|                140|       0|         0.0|
|54.0|           0|            0|        No Info|27.32|        6.6|                 80|       0|         0.0|
|28.0|           0|            0|          never|27.32|        5.7|                158|       0|         1.0|
|36.0|           0|            0|        current|23.45|        5.0|                155|       0|         0.0|
|76.0|           1|            1|        current|20.14|        4.8|                155|       0|         1.0|
|20.0|           0|            0|          never|27.32|        6.6|                 85|       0|         0.0|
|44.0|    

In [18]:
from pyspark.ml.feature import StringIndexer
indexer=StringIndexer(
    inputCol="smoking_history",
    outputCol="smoking_history_indexer"
)
df=indexer.fit(df).transform(df)
df.show()

+----+------------+-------------+---------------+-----+-----------+-------------------+--------+------------+-----------------------+
| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|diabetes|gender_index|smoking_history_indexer|
+----+------------+-------------+---------------+-----+-----------+-------------------+--------+------------+-----------------------+
|80.0|           0|            1|          never|25.19|        6.6|                140|       0|         0.0|                    1.0|
|54.0|           0|            0|        No Info|27.32|        6.6|                 80|       0|         0.0|                    0.0|
|28.0|           0|            0|          never|27.32|        5.7|                158|       0|         1.0|                    1.0|
|36.0|           0|            0|        current|23.45|        5.0|                155|       0|         0.0|                    3.0|
|76.0|           1|            1|        current|20.14|       

In [19]:
df=df.drop("smoking_history")
df.show()

+----+------------+-------------+-----+-----------+-------------------+--------+------------+-----------------------+
| age|hypertension|heart_disease|  bmi|HbA1c_level|blood_glucose_level|diabetes|gender_index|smoking_history_indexer|
+----+------------+-------------+-----+-----------+-------------------+--------+------------+-----------------------+
|80.0|           0|            1|25.19|        6.6|                140|       0|         0.0|                    1.0|
|54.0|           0|            0|27.32|        6.6|                 80|       0|         0.0|                    0.0|
|28.0|           0|            0|27.32|        5.7|                158|       0|         1.0|                    1.0|
|36.0|           0|            0|23.45|        5.0|                155|       0|         0.0|                    3.0|
|76.0|           1|            1|20.14|        4.8|                155|       0|         1.0|                    3.0|
|20.0|           0|            0|27.32|        6.6|     

In [20]:
df.columns

['age',
 'hypertension',
 'heart_disease',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'diabetes',
 'gender_index',
 'smoking_history_indexer']

In [21]:
feature=['age',
 'hypertension',
 'heart_disease',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'gender_index',
 'smoking_history_indexer']

In [22]:
df.printSchema()

root
 |-- age: double (nullable = false)
 |-- hypertension: integer (nullable = true)
 |-- heart_disease: integer (nullable = true)
 |-- bmi: double (nullable = false)
 |-- HbA1c_level: double (nullable = true)
 |-- blood_glucose_level: integer (nullable = true)
 |-- diabetes: integer (nullable = true)
 |-- gender_index: double (nullable = false)
 |-- smoking_history_indexer: double (nullable = false)



In [23]:
from pyspark.ml.feature import VectorAssembler
assembler=VectorAssembler(
    inputCols= feature,
    outputCol="column_features"
)
df=assembler.transform(df)

In [24]:
df.show()

+----+------------+-------------+-----+-----------+-------------------+--------+------------+-----------------------+--------------------+
| age|hypertension|heart_disease|  bmi|HbA1c_level|blood_glucose_level|diabetes|gender_index|smoking_history_indexer|     column_features|
+----+------------+-------------+-----+-----------+-------------------+--------+------------+-----------------------+--------------------+
|80.0|           0|            1|25.19|        6.6|                140|       0|         0.0|                    1.0|[80.0,0.0,1.0,25....|
|54.0|           0|            0|27.32|        6.6|                 80|       0|         0.0|                    0.0|(8,[0,3,4,5],[54....|
|28.0|           0|            0|27.32|        5.7|                158|       0|         1.0|                    1.0|[28.0,0.0,0.0,27....|
|36.0|           0|            0|23.45|        5.0|                155|       0|         0.0|                    3.0|[36.0,0.0,0.0,23....|
|76.0|           1|        

# Train-Test Split

In [25]:
train_df,test_df=df.randomSplit([0.8,0.2],seed=42)

# Train Model (Logistic Regression)

In [26]:
from pyspark.ml.classification import LogisticRegression
model=LogisticRegression(
    featuresCol="column_features" ,
    labelCol="diabetes")
model_trained=model.fit(train_df)

# Predictions

In [27]:
prediction=model_trained.transform(test_df)
prediction.select("column_features","diabetes","prediction")

DataFrame[column_features: vector, diabetes: int, prediction: double]

# Accuracy Check

In [28]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator=BinaryClassificationEvaluator(labelCol="diabetes")
accuracy=evaluator.evaluate(prediction)
print("Accuracy",accuracy)

Accuracy 0.9618887597902482
